# Search and A* Laboratory Exercise
## Using an LLM as an Engineering Assistant

### Task 0: Understand the Search Problem

**Formulation of the warehouse problem:**
- **State $S$**: The $(r, c)$ coordinates of the robot in the 2D grid.
- **Actions $A$**: Up, Down, Left, Right.
- **Transition $T$**: $(r,c) \xrightarrow{\text{action}} (r+dr, c+dc)$ if the target cell is not an obstacle and is within bounds.
- **Initial state $s_0$**: The $(r, c)$ coordinates of the starting position 'S'.
- **Goal $G$**: The $(r, c)$ coordinates of the goal position 'G'.
- **Cost $c$**: 1 for every movement step.

**(a) What information is necessary to specify a state?**
The current $(row, column)$ coordinates of the robot in the grid.

**(b) What makes an action invalid?**
An action is invalid if it leads the robot outside the boundaries of the warehouse map, or if the destination cell is an obstacle ('#').

**(c) Is this a deterministic search problem?**
Yes. Executing an action in a specific state always results in exactly one predictable next state.

**(d) What would constitute a solution?**
A sequence of valid actions (or a path of states) starting from $s_0$ and ending at $G$.

---

### Task 1: Plan the Agent

**Design for the search agent:**
1. **How a state will be represented in Python:** As a tuple of integers `(r, c)`.
2. **How the warehouse will be represented:** As a list of strings or a 2D list of characters.
3. **How valid actions will be determined:** By defining a list of direction vectors `[(-1,0), (1,0), (0,-1), (0,1)]` and checking if the resulting position is within grid boundaries and `!= '#'`.
4. **How the agent will recognise that it has reached the goal:** By checking if the current state `(r, c) == goal_state`.
5. **What information must be stored in the frontier:** For A*, a priority queue (min-heap) storing tuples of `(f_score, g_score, current_state, path_so_far)`.
6. **How the final path will be reconstructed:** By storing the path as a list alongside the state in the frontier, or by keeping a `came_from` dictionary mapping each state to its predecessor.

---

### Task 2: Ask an LLM to Generate A*

**Prompt used:**
> I am implementing a simple goal-based search agent in Python. The environment is a grid represented by an ASCII map. The agent starts at S and must reach G. # represent obstacles and . represents free cells. The agent can move up, down, left, right (cost 1). Implement A* search using Manhattan distance as the heuristic. The program should represent grid positions as states, maintain a frontier, calculate g, h, f, avoid repeatedly expanding the same state, reconstruct the path, and report the path, its length, and the number of states expanded.



In [ ]:
import heapq
from collections import deque
import math

warehouse_map = [
    "#################",
    "#S....#.........#",
    "#.###.#.#######.#",
    "#...#.#.......#.#",
    "###.#.#######.#.#",
    "#...#.........#.#",
    "#.###########.#.#",
    "#.............#G#",
    "#################"
]

def find_positions(grid):
    start, goal = None, None
    for r, row in enumerate(grid):
        for c, char in enumerate(row):
            if char == 'S': start = (r, c)
            elif char == 'G': goal = (r, c)
    return start, goal

def manhattan_heuristic(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def a_star_search(grid, heuristic_func=manhattan_heuristic, heuristic_weight=1.0):
    start, goal = find_positions(grid)
    rows, cols = len(grid), len(grid[0])
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    
    # Priority Queue: (f_score, g_score, current_state)
    frontier = []
    heapq.heappush(frontier, (0, 0, start))
    
    # Dictionary to keep track of the shortest path to each node
    came_from = {start: None}
    cost_so_far = {start: 0}
    states_expanded = 0
    
    while frontier:
        _, current_g, current = heapq.heappop(frontier)
        states_expanded += 1
        
        if current == goal:
            # Reconstruct path
            path = []
            while current is not None:
                path.append(current)
                current = came_from[current]
            path.reverse()
            return True, path, len(path) - 1, states_expanded
            
        for dr, dc in directions:
            nxt = (current[0] + dr, current[1] + dc)
            
            if 0 <= nxt[0] < rows and 0 <= nxt[1] < cols and grid[nxt[0]][nxt[1]] != '#':
                new_cost = current_g + 1
                
                if nxt not in cost_so_far or new_cost < cost_so_far[nxt]:
                    cost_so_far[nxt] = new_cost
                    f_score = new_cost + heuristic_weight * heuristic_func(nxt, goal)
                    heapq.heappush(frontier, (f_score, new_cost, nxt))
                    came_from[nxt] = current
                    
    return False, [], 0, states_expanded

def print_result(name, result):
    found, path, length, expanded = result
    print(f"--- {name} ---")
    print(f"Solution found: {found}")
    if found:
        print(f"Path length: {length}")
    print(f"States expanded: {expanded}\n")

res_astar_orig = a_star_search(warehouse_map)
print_result("A* on Original Warehouse", res_astar_orig)



### Task 3: Test the Generated Program

We will now run the generated A* code against multiple test scenarios.



In [ ]:
# Test 2: Trivial case
trivial_map = [
    "#####",
    "#SG##",
    "#####"
]
print_result("A* on Trivial Map", a_star_search(trivial_map))

# Test 3: No solution
no_sol_map = [
    "#######",
    "#S....#",
    "###.###",
    "#...#G#",
    "#######"
]
print_result("A* on No Solution Map", a_star_search(no_sol_map))

# Test 4: Alternative paths
alt_paths_map = [
    "#######",
    "#S...G#",
    "#.#.###",
    "#.....#",
    "#######"
]
print_result("A* on Alternative Paths Map", a_star_search(alt_paths_map))



### Task 4: Inspect the A* Algorithm

**Where concepts appear in the code:**
- **State**: Represented as `(r, c)` tuples.
- **Action**: Iterating over `directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]`.
- **Transition**: `nxt = (current[0] + dr, current[1] + dc)`.
- **Goal test**: `if current == goal:`.
- **$g(n)$**: Tracked via `new_cost = current_g + 1` and `cost_so_far` dictionary.
- **$h(n)$**: Calculated via `heuristic_func(nxt, goal)`.
- **$f(n)$**: Calculated as `f_score = new_cost + ...`.
- **Frontier**: `frontier = []` managed by `heapq.heappush` and `heapq.heappop`.
- **Visited states**: Implemented implicitly using `cost_so_far` which acts as both the distance table and the closed/visited set.
- **Path reconstruction**: The `while current is not None:` loop tracing backwards through the `came_from` dictionary.

**(a) What data structure is used for the A* frontier?**
A priority queue, specifically a min-heap implementation using Python's `heapq` module.

**(b) How does the program select the next state to expand?**
By popping the element with the lowest $f(n)$ score from the priority queue (`heapq.heappop`).

**(c) Where is the heuristic calculated?**
In the `manhattan_heuristic` helper function, which computes the absolute differences between X and Y coordinates.

**(d) Does the program explicitly calculate $f(n) = g(n) + h(n)$?**
Yes, in the line: `f_score = new_cost + heuristic_weight * heuristic_func(nxt, goal)`.

**(e) How does the program prevent unnecessary repeated exploration?**
By maintaining a `cost_so_far` dictionary. If a state has been seen before, it is only re-added to the frontier if the newly discovered path to it is strictly cheaper (shorter) than the previously known best path.



### Task 5: Compare A* with Blind Search
We will implement BFS and compare it against A* on the original warehouse map.



In [ ]:
def bfs_search(grid):
    start, goal = find_positions(grid)
    rows, cols = len(grid), len(grid[0])
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    
    queue = deque([start])
    came_from = {start: None}
    states_expanded = 0
    
    while queue:
        current = queue.popleft()
        states_expanded += 1
        
        if current == goal:
            path = []
            while current is not None:
                path.append(current)
                current = came_from[current]
            return True, path[::-1], len(path) - 1, states_expanded
            
        for dr, dc in directions:
            nxt = (current[0] + dr, current[1] + dc)
            if 0 <= nxt[0] < rows and 0 <= nxt[1] < cols and grid[nxt[0]][nxt[1]] != '#':
                if nxt not in came_from:
                    came_from[nxt] = current
                    queue.append(nxt)
                    
    return False, [], 0, states_expanded

res_bfs = bfs_search(warehouse_map)
print_result("BFS on Original Warehouse", res_bfs)



**(a) Did both algorithms find a solution?** Yes.
**(b) Did they find paths of the same length?** Yes, both found an optimal path of length 40.
**(c) Which algorithm expanded fewer states?** In this specific, highly constrained maze layout with narrow corridors, both A* and BFS were forced to explore exactly 64 states.
**(d) Why might A* expand fewer states?** While they tied on this specific map due to lack of alternative open paths, in an open room A* expands vastly fewer states. BFS expands uniformly in all directions. A* uses the heuristic to "pull" the search direction towards the goal, avoiding exploring dead ends that lead away from the goal unless absolutely necessary.



### Task 6: Investigate the Heuristic
**Why is Manhattan appropriate?** Because the robot can only move horizontally and vertically (no diagonals). The Manhattan distance perfectly estimates the true minimum cost in an empty grid.

Let's test $h(n) = 0$, Euclidean distance, and $h(n) \times 2$.



In [ ]:
def h_zero(a, b): return 0
def euclidean_heuristic(a, b): return math.sqrt((a[0] - b[0])**2 + (a[1] - b[1])**2)

print_result("A* with h(n) = 0 (Dijkstra)", a_star_search(warehouse_map, heuristic_func=h_zero))
print_result("A* with Euclidean Heuristic", a_star_search(warehouse_map, heuristic_func=euclidean_heuristic))
print_result("A* with 2 * Manhattan (Greedy/Weighted)", a_star_search(warehouse_map, heuristic_weight=2.0))



**Think About It (Admissibility)**
- If the heuristic is $0$, it behaves exactly like BFS/Dijkstra. It expands the maximum number of states but guarantees the shortest path.
- If the heuristic is Euclidean, it still guarantees the shortest path (because Euclidean $\le$ Manhattan, so it is admissible), but it is a "looser" bound, meaning it is less informed and expands slightly more states than Manhattan.
- If the heuristic is multiplied by 2 ($h(n) \times 2$), it becomes **inadmissible** (it overestimates the cost). It becomes very aggressive, acting greedily. It expands the absolute fewest states, but sacrifices the guarantee of finding the optimal shortest path (it might find a longer, suboptimal path).



### Task 7 & Final Reflection

**Task 7: Evaluate the LLM-Generated Agent**
1. The standard A* loop, priority queue logic, and grid bounds checking were correct immediately.
2. The LLM correctly used a `came_from` dictionary for path reconstruction, which is efficient.
3. The LLM clearly structured the code, separating the search algorithm from the map representation.
4. I tested the trivial case, the no-solution case, and the complex maze. These systematic tests were highly useful to guarantee that the LLM didn't accidentally hardcode assumptions or create infinite loops.
5. Trusting the program without testing it is dangerous. LLMs can easily introduce off-by-one errors (e.g. counting the start state in the path length) or forget to handle the case where the priority queue empties out before the goal is found.

**Final Reflection**
1. **Why is it important to formulate the search problem before writing the search algorithm?** Because the search algorithm (BFS, A*) is just an abstract engine. Without precisely defining the states, valid actions, and goal tests, you cannot translate the physical problem (the warehouse) into the mathematical model the algorithm expects.
2. **In what sense is A* an "informed" search algorithm?** It is informed because it uses problem-specific knowledge (the heuristic distance to the goal) to prioritize which states to explore next, unlike blind searches which only know the cost from the start.
3. **Why does the choice of heuristic matter?** The heuristic controls the trade-off between speed and optimality. A perfectly tight, admissible heuristic explores the minimum possible states while guaranteeing the shortest path. An inadmissible heuristic might be fast but can yield terrible, suboptimal paths.
4. **What did the LLM contribute to the engineering process?** It contributed rapid prototyping, handling the tedious boilerplate of setting up priority queues, dictionaries, and bounds-checking logic, allowing the engineer to focus on the high-level design and validation.
5. **What could go wrong if an engineer simply accepted LLM-generated code without testing it?** The code might contain subtle logical bugs (like incorrect path length calculation), it might loop infinitely on unsolvable maps, or it might use an inadmissible heuristic by mistake, causing autonomous robots in production to take horribly inefficient paths. Working output is not a validated algorithm.

